# 02. 주식 트레이딩 PPO (Sim-to-Finance)
> **Google Colab T4** 에서 위에서 아래로 순서대로 셀을 실행하세요.

## 1. 패키지 설치

In [ ]:
!apt-get install -y swig > /dev/null 2>&1
!pip install -q stable-baselines3==2.3.2 gymnasium==0.29.1 shimmy==1.3.0 tensorboard>=2.9.1 yfinance>=0.2.50

## 2. 모듈 작성 및 경로 등록

In [ ]:
import os, sys
os.makedirs('/content/rl_trading', exist_ok=True)
if '/content/rl_trading' not in sys.path:
    sys.path.insert(0, '/content/rl_trading')
os.makedirs('/content/results/models', exist_ok=True)
os.makedirs('/content/results/plots', exist_ok=True)
print('디렉토리 준비 완료')

In [ ]:
%%writefile /content/rl_trading/fetcher.py
import yfinance as yf
import pandas as pd
import numpy as np
from typing import Tuple


def fetch_stock_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """yf.Ticker().history() 방식으로 OHLCV 수집 — yf.download()보다 API 변경에 강함."""
    tk = yf.Ticker(ticker)
    df = tk.history(start=start, end=end, auto_adjust=True)
    if df.empty:
        raise ValueError(f"No data for '{ticker}' ({start}~{end}). 티커 확인 또는 잠시 후 재시도하세요.")
    df = df[["Close", "Volume"]].dropna()
    df.index = pd.to_datetime(df.index).tz_localize(None)  # tz 제거로 downstream 호환
    return df


def add_features(df: pd.DataFrame, window: int = 20) -> pd.DataFrame:
    """Append rolling-mean and rolling-std features used by the trading env."""
    df = df.copy()
    df["MA"] = df["Close"].rolling(window).mean()
    df["Std"] = df["Close"].rolling(window).std()
    df["Return"] = df["Close"].pct_change()
    df = df.dropna()
    return df


def normalize_prices(df: pd.DataFrame) -> Tuple[pd.DataFrame, float]:
    """Min-max normalize Close price; return scaled df and scale factor."""
    scale = float(df["Close"].max())
    df = df.copy()
    df["Close"] = df["Close"] / scale
    return df, scale


def train_test_split_df(df: pd.DataFrame, test_ratio: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame]:
    split = int(len(df) * (1 - test_ratio))
    return df.iloc[:split].copy(), df.iloc[split:].copy()

In [ ]:
%%writefile /content/rl_trading/trading_env.py
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class TradingEnv(gym.Env):
    """
    커스텀 주식 트레이딩 환경 (gymnasium 호환)

    상태 (observation):
        - 최근 window_size일 종가 (정규화)
        - 현재 보유 주식 수 (정규화)
        - 현재 현금 잔고 (정규화)

    행동 (action):
        0 = 홀드, 1 = 매수 (전량), 2 = 매도 (전량)

    보상:
        스텝마다 포트폴리오 총가치 변화율
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, prices: np.ndarray, window_size: int = 20, initial_balance: float = 10_000.0):
        super().__init__()
        assert len(prices) > window_size, "prices 길이가 window_size보다 커야 합니다."

        self.prices = prices.astype(np.float32)
        self.window_size = window_size
        self.initial_balance = float(initial_balance)

        obs_dim = window_size + 2  # 종가 window + 보유량 + 잔고
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)

        self._max_price = float(self.prices.max())
        self.reset()

    # ------------------------------------------------------------------
    # 내부 헬퍼
    # ------------------------------------------------------------------

    def _get_obs(self) -> np.ndarray:
        window = self.prices[self.current_step - self.window_size: self.current_step]
        norm_window = window / self._max_price
        norm_shares = np.array([self.shares_held / 100.0], dtype=np.float32)
        norm_cash = np.array([self.cash_balance / self.initial_balance], dtype=np.float32)
        return np.concatenate([norm_window, norm_shares, norm_cash]).astype(np.float32)

    def _portfolio_value(self) -> float:
        return self.cash_balance + self.shares_held * float(self.prices[self.current_step])

    # ------------------------------------------------------------------
    # gymnasium API
    # ------------------------------------------------------------------

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.cash_balance = self.initial_balance
        self.shares_held = 0
        self.prev_portfolio_value = self.initial_balance
        self.trade_log: list[dict] = []
        return self._get_obs(), {}

    def step(self, action: int):
        price = float(self.prices[self.current_step])

        if action == 1 and self.cash_balance >= price:  # 매수
            shares_to_buy = int(self.cash_balance // price)
            self.shares_held += shares_to_buy
            self.cash_balance -= shares_to_buy * price
            self.trade_log.append({"step": self.current_step, "action": "BUY", "price": price})

        elif action == 2 and self.shares_held > 0:  # 매도
            self.cash_balance += self.shares_held * price
            self.trade_log.append({"step": self.current_step, "action": "SELL", "price": price, "shares": self.shares_held})
            self.shares_held = 0

        portfolio_value = self._portfolio_value()
        reward = (portfolio_value - self.prev_portfolio_value) / self.prev_portfolio_value
        self.prev_portfolio_value = portfolio_value

        self.current_step += 1
        terminated = self.current_step >= len(self.prices) - 1
        truncated = False

        return self._get_obs(), float(reward), terminated, truncated, {"portfolio_value": portfolio_value}

    def render(self, mode="human"):
        price = float(self.prices[self.current_step])
        pv = self._portfolio_value()
        print(f"Step {self.current_step:4d} | Price: {price:8.2f} | "
              f"Shares: {self.shares_held:4d} | Cash: {self.cash_balance:10.2f} | Portfolio: {pv:10.2f}")

In [ ]:
%%writefile /content/rl_trading/ppo_agent.py
import os
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor


class RewardLoggerCallback(BaseCallback):
    """에피소드 종료마다 포트폴리오 가치를 기록하는 콜백."""

    def __init__(self):
        super().__init__()
        self.episode_rewards: list[float] = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
        return True


class PPOTradingAgent:
    """stable-baselines3 PPO를 트레이딩 환경에 맞게 래핑한 클래스."""

    def __init__(
        self,
        env_fn,
        policy: str = "MlpPolicy",
        learning_rate: float = 3e-4,
        n_steps: int = 2048,
        batch_size: int = 64,
        n_epochs: int = 10,
        gamma: float = 0.99,
        verbose: int = 0,
        device: str = "auto",
    ):
        self.env = DummyVecEnv([lambda: Monitor(env_fn())])
        self.model = PPO(
            policy,
            self.env,
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            n_epochs=n_epochs,
            gamma=gamma,
            verbose=verbose,
            device=device,
        )
        self.reward_callback = RewardLoggerCallback()

    def train(self, total_timesteps: int = 100_000):
        self.model.learn(total_timesteps=total_timesteps, callback=self.reward_callback)

    def predict(self, obs: np.ndarray) -> int:
        action, _ = self.model.predict(obs, deterministic=True)
        return int(action)

    def save(self, path: str):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        self.model.save(path)

    def load(self, path: str):
        self.model = PPO.load(path, env=self.env)

    def evaluate(self, env, n_episodes: int = 5) -> list[float]:
        """env 위에서 n_episodes 실행 후 에피소드별 포트폴리오 최종 가치를 반환."""
        portfolio_values = []
        for _ in range(n_episodes):
            obs, _ = env.reset()
            done = False
            while not done:
                action = self.predict(obs)
                obs, _, terminated, truncated, info = env.step(action)
                done = terminated or truncated
            portfolio_values.append(info.get("portfolio_value", 0.0))
        return portfolio_values

In [ ]:
%%writefile /content/rl_trading/visualizer.py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from typing import Optional


def plot_learning_curve(
    rewards: list[float],
    title: str = "Learning Curve",
    window: int = 50,
    save_path: Optional[str] = None,
):
    """에피소드 보상 + 이동평균 학습 곡선."""
    fig, ax = plt.subplots(figsize=(10, 4))
    episodes = np.arange(1, len(rewards) + 1)
    ax.plot(episodes, rewards, alpha=0.3, color="steelblue", label="Episode reward")

    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
        ax.plot(np.arange(window, len(rewards) + 1), ma, color="steelblue", linewidth=2, label=f"MA-{window}")

    ax.set_xlabel("Episode")
    ax.set_ylabel("Total Reward")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def plot_portfolio(
    prices: np.ndarray,
    portfolio_values: np.ndarray,
    trade_log: list[dict],
    ticker: str = "",
    initial_balance: float = 10_000.0,
    save_path: Optional[str] = None,
):
    """주가 + 포트폴리오 가치 + 매수/매도 시점을 2패널로 시각화."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    steps = np.arange(len(prices))

    # 상단: 주가 + 매수/매도 마커
    ax1.plot(steps, prices, color="black", linewidth=1, label="Price")
    buys = [t for t in trade_log if t["action"] == "BUY"]
    sells = [t for t in trade_log if t["action"] == "SELL"]
    if buys:
        ax1.scatter([t["step"] for t in buys], [t["price"] for t in buys],
                    marker="^", color="green", s=60, zorder=5, label="BUY")
    if sells:
        ax1.scatter([t["step"] for t in sells], [t["price"] for t in sells],
                    marker="v", color="red", s=60, zorder=5, label="SELL")
    ax1.set_ylabel("Price")
    ax1.set_title(f"{ticker} — Trading Result" if ticker else "Trading Result")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)

    # 하단: 포트폴리오 가치
    ax2.plot(steps[:len(portfolio_values)], portfolio_values, color="royalblue", linewidth=1.5, label="Portfolio")
    ax2.axhline(initial_balance, color="gray", linestyle="--", linewidth=1, label="Initial balance")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Portfolio Value ($)")
    ax2.legend(loc="upper left")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def plot_action_distribution(trade_log: list[dict], save_path: Optional[str] = None):
    """매수/매도/홀드 비율 파이차트."""
    total = max(len(trade_log), 1)
    buy_n = sum(1 for t in trade_log if t["action"] == "BUY")
    sell_n = sum(1 for t in trade_log if t["action"] == "SELL")
    hold_n = total - buy_n - sell_n

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(
        [buy_n, sell_n, hold_n],
        labels=["BUY", "SELL", "HOLD"],
        colors=["green", "red", "gray"],
        autopct="%1.1f%%",
        startangle=90,
    )
    ax.set_title("Action Distribution")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig

## 3. 주가 데이터 수집

In [ ]:
from fetcher import fetch_stock_data, add_features, train_test_split_df
import numpy as np

TICKER = 'AAPL'   # 원하는 티커로 변경 가능 (예: '005930.KS')
START  = '2018-01-01'
END    = '2024-01-01'

df_raw = fetch_stock_data(TICKER, START, END)
df     = add_features(df_raw, window=20)
df_train, df_test = train_test_split_df(df, test_ratio=0.2)

print(f'학습 데이터: {len(df_train)}일  |  테스트 데이터: {len(df_test)}일')
df.tail(3)

## 4. 트레이딩 환경 확인

In [ ]:
from trading_env import TradingEnv

prices_train = df_train['Close'].values.astype('float32')
env = TradingEnv(prices_train, window_size=20, initial_balance=10_000)

obs, _ = env.reset()
print('Observation shape:', obs.shape)
print('Action space:', env.action_space)

done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
print(f'랜덤 에이전트 최종 포트폴리오: ${info["portfolio_value"]:,.2f}')

## 5. PPO 학습

In [ ]:
from ppo_agent import PPOTradingAgent

agent = PPOTradingAgent(
    env_fn=lambda: TradingEnv(prices_train, window_size=20, initial_balance=10_000),
    learning_rate=3e-4, n_steps=512, batch_size=64, n_epochs=10,
    verbose=1, device='auto',
)
agent.train(total_timesteps=200_000)
agent.save('/content/results/models/ppo_trading')
print('학습 완료')

## 6. 학습 곡선 시각화

In [ ]:
from visualizer import plot_learning_curve

fig = plot_learning_curve(
    agent.reward_callback.episode_rewards,
    title=f'{TICKER} PPO Learning Curve', window=30,
    save_path='/content/results/plots/trading_curve.png',
)
fig.show()

## 7. 테스트 데이터로 백테스트

In [ ]:
from visualizer import plot_portfolio, plot_action_distribution

prices_test = df_test['Close'].values.astype('float32')
test_env = TradingEnv(prices_test, window_size=20, initial_balance=10_000)

obs, _ = test_env.reset()
done = False
portfolio_values = [10_000.0]

while not done:
    action = agent.predict(obs)
    obs, _, terminated, truncated, info = test_env.step(action)
    portfolio_values.append(info['portfolio_value'])
    done = terminated or truncated

final_value = portfolio_values[-1]
ret = (final_value - 10_000) / 10_000 * 100
print(f'테스트 수익률: {ret:+.2f}%  |  최종 포트폴리오: ${final_value:,.2f}')

fig1 = plot_portfolio(prices_test, np.array(portfolio_values), test_env.trade_log,
    ticker=TICKER, initial_balance=10_000,
    save_path='/content/results/plots/trading_result.png')
fig1.show()

fig2 = plot_action_distribution(test_env.trade_log,
    save_path='/content/results/plots/action_dist.png')
fig2.show()

---
다음 → **app/streamlit_app.py** 를 로컬 또는 Colab 터널링으로 실행해 인터랙티브 데모를 확인하세요.